In [ ]:
!pip install openai pandas tqdm # для запросов к моделям через OpenRouter

import os
import time
import random
import pandas as pd

from tqdm import tqdm
from openai import OpenAI
from getpass import getpass # для безопасного ввода API-ключа.

In [ ]:
# ввожу ключ от OpenRouter
os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

OpenRouter API key: ··········


In [ ]:
# client через которого идет обращение к моделям

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
print(os.environ["OPENROUTER_API_KEY"][:20])

sk-or-v1-818db7347f3


In [ ]:
# загрузка датасета настоящих ученических текстов

df = pd.read_csv(
    "human_texts.csv",
    sep=";"
)
df["id"] = range(len(df))
df = df.rename(columns={
    "Текст": "text", "Год": "year"
})
df = df.drop(columns=["Источник"])
df["id"] = df["id"].astype(str)

In [ ]:
MODELS = [
    "google/gemini-3.1-flash-lite",
    "qwen/qwen-2.5-72b-instruct",
    "openai/gpt-4o-mini"
]

In [ ]:
# Для разнообразия генерации делаем несколько типов промптов

PROMPTS = {
    #  парафразирование сочинения
    "paraphrase": """
Перепиши это сочинение так, как будто его писал обычный школьник.
Не делай текст слишком правильным или литературным.
Допустимы:
- небольшие повторы,
- разговорные фразы,
- эмоциональные комментарии,
- местами неровная логика.
Текст не должен выглядеть как образцовое сочинение.
Сохрани общий смысл, но сделай текст более живым и естественным.

{text}
""",

    # улучшение текста
    "improve": """
Улучши сочинение школьника,
но сохрани человеческий стиль и простые формулировки. Структуру блоков сочинения.

{text}
""",

    # стиль школьника
    "student_style": """
Напиши сочинение как обычный подросток.
Не делай идеальную структуру.
Некоторые мысли могут повторяться или быть сформулированы не очень аккуратно.
Допустимы:
- разговорные слова,
- бытовые примеры,
- личные впечатления,
- эмоциональные реакции.
Текст не должен выглядеть как работа отличника или как ответ языковой модели.

{text}
""",

    #  сделать текст менее палевным
    "adversarial": """
Перепиши текст так, чтобы опытный преподаватель поверил, что его написал школьник.
Избегай:
- слишком правильной структуры,
- идеально связанных абзацев,
- академического стиля,
- шаблонных выводов.
Добавь:
- человеческие шероховатости,
- эмоциональные вставки,
- слегка неровный ритм,
- бытовые детали,
- местами неидеальные формулировки.
Текст должен выглядеть естественно, а не идеально.
Не делай его слишком умным или литературным.
Перепиши текст так,
чтобы он выглядел написанным учеником, а не языковой моделью.

{text}
"""
}

In [ ]:
# функция для генерации текста

def generate_text(model, prompt):

    response = client.chat.completions.create(
        model=model,
        messages=[
            # объясняю модели кто она и что делать
            {
                "role": "system",
                "content": (
                    "Ты пишешь русскоязычное школьное сочинение от лица обычного ученика. "
                    "Текст должен выглядеть естественно и по-человечески, а не как идеально написанное сочинение. "

                    "Не делай текст слишком логичным, гладким и академичным. "
                    "Допустимы небольшие повторы мыслей, неровные переходы между абзацами, "
                    "эмоциональные комментарии, разговорные фразы и немного корявые формулировки. "

                    "Иногда ученик может отвлекаться на личные мысли или бытовые ассоциации. "
                    "Не нужно делать идеально симметричную структуру аргументов. "
                    "Не старайся делать каждый абзац одинаково сильным и завершенным. "

                    "Избегай типичных GPT-фраз: "
                    "«таким образом», "
                    "«следовательно», "
                    "«можно сделать вывод», "
                    "«автор хотел показать», "
                    "«данный пример подтверждает». "

                    "Не делай текст слишком связным и литературным. "
                    "Текст должен ощущаться написанным живым подростком, а не языковой моделью. "

                    "Никаких комментариев от имени ИИ. "
                    "Только текст сочинения."
                    )
            },
            # основной запрос
            {
                "role": "user",
                "content": prompt
            }


        ],

        # Генерируем с разной температурой для более разнообразных текстов
        temperature=random.choice([0.7, 0.8, 0.9]),
        top_p=0.9,
        max_tokens=1800    # максимальная длина ответа
    )


    return response.choices[0].message.content.strip()  #текст ответа

In [ ]:
# Генерация сочинений
results = [] # итоговый сгенерированный датасет

for _, row in tqdm(df.iterrows(), total=len(df)):

    model = random.choice(MODELS)       # случайно модель для генерации текста
    source_id = row["id"]   # id исходного текста чтобы разделить тест и трейн
    source_text = row["text"]
    prompt_type = random.choice(list(PROMPTS.keys()))

    prompt = PROMPTS[prompt_type].format(
        text=source_text
    )

    try:

        generated = generate_text(
            model=model,
            prompt=prompt
        )

        results.append({
            "source_id": source_id,
            "text": generated,
            "label": "ai",
            "model": model,
            "prompt_type": prompt_type
        })

        time.sleep(1)

    except Exception as e:

        print("Ошибка:", e)

 33%|███▎      | 405/1236 [1:14:31<1:43:19,  7.46s/it]

Ошибка: 'NoneType' object is not subscriptable


 33%|███▎      | 409/1236 [1:14:47<1:00:38,  4.40s/it]

Ошибка: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{"code":400,"reason":"INVALID_REQUEST_BODY","message":"model: qwen/qwen-2.5-72b-instruct does not support endpoint: completions","metadata":{}}', 'provider_name': 'Novita', 'is_byok': False}}, 'user_id': 'user_3DUHR6OtBKUiGaVqP1L1tNQffWg'}


 33%|███▎      | 411/1236 [1:14:55<54:22,  3.95s/it]  

...

 87%|████████▋ | 1075/1236 [3:25:24<37:56, 14.14s/it]

Ошибка: 'NoneType' object is not subscriptable


 90%|████████▉ | 1109/1236 [3:33:22<28:56, 13.68s/it]

Ошибка: 'NoneType' object is not subscriptable


100%|██████████| 1236/1236 [3:59:12<00:00, 11.61s/it]


In [ ]:
AI_df = pd.DataFrame(results)
AI_df.to_csv(
    "synthetic_texts.csv",
    index=False,
    encoding="utf-8-sig"
)
print("Генерация готова,тексты сохранены")

Генерация готова,тексты сохранены


In [ ]:
AI_df = pd.read_csv(
    "synthetic_texts.csv"
)


In [ ]:
# немного переделаем файл с человеческими текстами для объединения

human_df = df.copy()

human_df["source_id"] = human_df["id"]
human_df["label"] = "human"
human_df["model"] = "human"
human_df["prompt_type"] = "none"

human_df = human_df[
    ["source_id", "text", "label", "model", "prompt_type"]
]

In [ ]:
# объединяем школьные + ai

final_df = pd.concat(
    [human_df, AI_df],
    ignore_index=True
)

final_df.to_csv(
    "final_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Финальный датасет готов")

Финальный датасет готов


In [ ]:
from google.colab import files
files.download("synthetic_texts.csv")
files.download("final_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# загружаю готовый csv

full_df = pd.read_csv(
    "final_dataset.csv"
)

# отдельно human и ai

human_df = full_df[
    full_df["label"] == "human"
]

ai_df = full_df[
    full_df["label"] == "ai"
]

# беру по 50 случайных текстов

human_sample = human_df.sample(
    50,
    random_state=42
)

ai_sample = ai_df.sample(
    50,
    random_state=42
)

# объединяю

train_100 = pd.concat([
    human_sample,
    ai_sample
])

# перемешиваю

train_100 = train_100.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# проверка

print(
    train_100["label"].value_counts()
)

# сохраняю

train_100.to_csv(
    "train_100_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)

train_100.head()

label
ai       50
human    50
Name: count, dtype: int64


,source_id,text,label,model,prompt_type
0,908,"Красота и доброта – это, казалось бы, два таки...",ai,openai/gpt-4o-mini,improve
1,1012,"Бунин — это, конечно, просто огонь в русской л...",ai,openai/gpt-4o-mini,paraphrase
2,434,"В произведениях, как ни крути, всегда есть как...",ai,openai/gpt-4o-mini,adversarial
3,101,Роман Б. Л. Пастернака «Доктор Живаго» рассказ...,human,human,none
4,308,"Драгоценные книги — это книги, которые связаны...",human,human,none
